# Intelligent Report Summarizer - Universal Edition

**Analyze 3 types of reports: Individual Stocks, Sector/Industry, Macro Economy**

**Uses: Llama 3.1 (Best quality, thorough analysis)**

## 📌 SETUP INSTRUCTIONS (First time only)

### 1. Install Ollama
Download from: https://ollama.ai

### 2. Download Llama 3.1
Open **Command Prompt/Terminal** and run:
```bash
ollama pull llama3.1
```
This downloads ~45GB (takes 15-30 minutes on good internet)

### 3. START OLLAMA SERVER (MUST DO THIS!)
**Open NEW Command Prompt/Terminal and run:**
```bash
ollama serve
```

**Wait for message:** `Listening on 127.0.0.1:11434`

**Keep this terminal window OPEN while using the notebook!**

### 4. Run This Notebook
1. Run **Cell 1** - Setup (checks Ollama and Llama 3.1)
2. Run **Cell 2** - Select type and enter inputs
3. Run **Cell 3** - PROCESSING (wait for completion - 5-10 min per report)
4. Run **Cell 4** - View results

---

## 📂 Report Types Supported

### Type 1: Individual Stock Reports
**Location:** `C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\CompanyResearchReports\FY*\*\`

**File pattern:** `FY26Q3_VENUSPIPES_AMBIT.pdf`

**You provide:**
- Ticker (e.g., VENUSPIPES)
- Broker (e.g., AMBIT)
- Last X **QUARTERS** (e.g., 4)

---

### Type 2: Sector/Industry Reports
**Location:** `C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\Sector\IndustryName\`

**File pattern:** `20260407_Banks_MOSL.pdf`

**Available sectors:** Auto, BFSI, Banking, Oil&Gas, Pharma, IT, etc.

**You provide:**
- Report Name (e.g., Banks, Oil&Gas)
- Broker (e.g., MOSL, AMBIT)
- Last X **REPORTS** (e.g., 3)

---

### Type 3: Macro Economy Reports
**Location:** `C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\MacroEconomy\`

**File pattern:** `20260416_EcoScope_MOSL.pdf`

**Report names:** EcoScope, Economy, MarketView, etc.

**You provide:**
- Report Name (e.g., EcoScope, Economy)
- Broker (e.g., MOSL, AMBIT)
- Last X **REPORTS** (e.g., 3)

---

## ⏱️ Timing

**Llama 3.1 (Only option):**
- ~5-10 minutes per report (thorough analysis)
- Can process 4 reports in 20-40 minutes
- Best quality summaries

---

In [43]:
import os, re, pandas as pd, json, warnings, requests
warnings.filterwarnings('ignore')
from pathlib import Path
from datetime import datetime
import time

try:
    import pdfplumber
except:
    os.system("pip install pdfplumber --break-system-packages")
    import pdfplumber

BASE_RESEARCH_PATH = Path(r'C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports')

# Test Ollama connection
OLLAMA_URL = "http://localhost:11434"
MODEL = "llama3.1"
TIMEOUT_SECONDS = 600  # 10 minutes - generous for Llama 3.1

try:
    response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
    if response.status_code == 200:
        models = response.json().get('models', [])
        available_models = [m['name'].split(':')[0] for m in models]
        print(f"✓ Ollama connected")
        print(f"\n📦 Available models: {', '.join(available_models).upper()}")
        
        if 'llama3.1' not in available_models and 'llama3' not in available_models:
            print(f"\n❌ ERROR: Llama 3.1 not found!")
            print(f"\n📌 Download it:")
            print(f"   ollama pull llama3.1")
            raise Exception("Llama 3.1 not installed")
        
        # Use Llama 3.1
        if 'llama3.1' in available_models:
            MODEL = "llama3.1"
        elif 'llama3' in available_models:
            MODEL = "llama3"
        
        print(f"  Using: {MODEL.upper()} (thorough analysis)")
        print(f"  Timeout: {TIMEOUT_SECONDS}s (10 minutes)")
except:
    print("❌ ERROR: Ollama not running!")
    print("\n📌 START OLLAMA SERVER:")
    print("   1. Open a NEW Command Prompt/Terminal")
    print("   2. Run: ollama serve")
    print("   3. Wait for: 'Listening on 127.0.0.1:11434'")
    print("   4. Keep that window OPEN")
    print("   5. Then run this notebook")
    raise

print("\n✓ Cell 1 Complete: Libraries ready, Ollama connected")

✓ Ollama connected

📦 Available models: LLAMA2, LLAMA3.1
  Using: LLAMA3.1 (thorough analysis)
  Timeout: 600s (10 minutes)

✓ Cell 1 Complete: Libraries ready, Ollama connected


In [45]:
print("\n" + "="*90)
print("INTELLIGENT REPORT SUMMARIZER - INPUT FORM")
print("="*90)

# Select report type
print("\n1️⃣  SELECT REPORT TYPE:")
print(f"   [1] Individual Stock (e.g., VENUSPIPES quarterly)")
print(f"   [2] Sector/Industry (e.g., Banks, Oil&Gas)")
print(f"   [3] Macro Economy (e.g., EcoScope, Economy)")
report_type = input("\nEnter 1, 2, or 3: ").strip()

if report_type == "1":
    REPORT_TYPE = "STOCK"
    label = "Ticker"
elif report_type == "2":
    REPORT_TYPE = "SECTOR"
    label = "Sector/Industry Name"
elif report_type == "3":
    REPORT_TYPE = "ECONOMY"
    label = "Report Name"
else:
    print("❌ Invalid choice!")
    raise ValueError("Must enter 1, 2, or 3")

# Get inputs
report_name = input(f"\n2️⃣  {label}: ").strip()
broker = input(f"3️⃣  Broker (AMBIT/MOSL/AXIS/etc): ").strip().upper()

if REPORT_TYPE == "STOCK":
    qty_input = input(f"4️⃣  Last X QUARTERS [default=4]: ").strip() or "4"
else:
    qty_input = input(f"4️⃣  Last X REPORTS [default=3]: ").strip() or "3"

qty_limit = int(qty_input)

CONFIG = {
    'type': REPORT_TYPE,
    'name': report_name,
    'broker': broker,
    'qty': qty_limit,
    'model': MODEL
}

print(f"\n" + "="*90)
print(f"✓ INPUTS ACCEPTED - Ready to process")
print(f"="*90)
print(f"  Type: {REPORT_TYPE}")
print(f"  Name: {report_name}")
print(f"  Broker: {broker}")
print(f"  Reports: {qty_limit}")
print(f"  Model: {MODEL.upper()}")
print(f"  Est. Time: ~{qty_limit * 8} minutes (thorough analysis)")
print(f"\n⚠️  This is thorough but slow. Each report takes 5-10 minutes.")
print(f"\n⏭️  NOW RUN CELL 3 (Processing) and come back later!")
print(f"="*90)


INTELLIGENT REPORT SUMMARIZER - INPUT FORM

1️⃣  SELECT REPORT TYPE:
   [1] Individual Stock (e.g., VENUSPIPES quarterly)
   [2] Sector/Industry (e.g., Banks, Oil&Gas)
   [3] Macro Economy (e.g., EcoScope, Economy)



Enter 1, 2, or 3:  1

2️⃣  Ticker:  ANUP
3️⃣  Broker (AMBIT/MOSL/AXIS/etc):  AMBIT
4️⃣  Last X QUARTERS [default=4]:  3



✓ INPUTS ACCEPTED - Ready to process
  Type: STOCK
  Name: ANUP
  Broker: AMBIT
  Reports: 3
  Model: LLAMA3.1
  Est. Time: ~24 minutes (thorough analysis)

⚠️  This is thorough but slow. Each report takes 5-10 minutes.

⏭️  NOW RUN CELL 3 (Processing) and come back later!


In [47]:
# ============================================================================
# BIG PROCESSING CELL - Type-aware report processing
# ============================================================================

def extract_pdf_text(pdf_path, max_chars=3500):
    """Extract text from PDF"""
    try:
        text_content = []
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            if total_pages > 0:
                page = pdf.pages[0]
                text = page.extract_text() or ""
                if text:
                    text_content.append(text)
            for page_num in range(max(1, total_pages - 4), total_pages):
                page = pdf.pages[page_num]
                text = page.extract_text() or ""
                if text:
                    text_content.append(text)
        combined_text = "\n".join(text_content)
        return combined_text[:max_chars]
    except:
        return ""

def call_llama(prompt):
    """Call Llama via Ollama API with 10-minute timeout"""
    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": MODEL,
                "prompt": prompt,
                "stream": False,
                "temperature": 0.7
            },
            timeout=TIMEOUT_SECONDS  # 10 minutes
        )
        if response.status_code == 200:
            return response.json()['response'].strip()
        else:
            return f"Error: HTTP {response.status_code}"
    except Exception as e:
        return f"Error: {str(e)}"

def find_stock_reports(ticker, broker):
    """Find individual stock reports (FY##Q#_TICKER_BROKER.pdf)"""
    reports = []
    company_path = BASE_RESEARCH_PATH / 'CompanyResearchReports'
    if not company_path.exists():
        return reports
    for fy_folder in company_path.glob('FY*'):
        if fy_folder.is_dir():
            for industry_folder in fy_folder.glob('*'):
                if industry_folder.is_dir():
                    for pdf_file in industry_folder.glob(f'*{ticker}*{broker}*.pdf'):
                        match = re.match(r'FY(\d{2})Q(\d)', pdf_file.name)
                        if match:
                            fy = int(match.group(1))
                            q = int(match.group(2))
                            reports.append({
                                'file': pdf_file,
                                'filename': pdf_file.name,
                                'key': f'FY{fy}Q{q}',
                                'sort_key': (fy, q)
                            })
    reports.sort(key=lambda x: x['sort_key'], reverse=True)
    return reports

def find_sector_reports(sector, broker):
    """Find sector reports (yyyymmdd_SectorName_Broker.pdf)"""
    reports = []
    sector_path = BASE_RESEARCH_PATH / 'Sector' / sector
    if not sector_path.exists():
        return reports
    for pdf_file in sector_path.glob(f'*{sector}*{broker}*.pdf'):
        match = re.match(r'(\d{8})', pdf_file.name)
        if match:
            date_str = match.group(1)
            reports.append({
                'file': pdf_file,
                'filename': pdf_file.name,
                'key': date_str,
                'sort_key': date_str
            })
    reports.sort(key=lambda x: x['sort_key'], reverse=True)
    return reports

def find_economy_reports(report_name, broker):
    """Find economy reports (yyyymmdd_ReportName_Broker.pdf)"""
    reports = []
    economy_path = BASE_RESEARCH_PATH / 'MacroEconomy'
    if not economy_path.exists():
        return reports
    for pdf_file in economy_path.glob(f'*{report_name}*{broker}*.pdf'):
        match = re.match(r'(\d{8})', pdf_file.name)
        if match:
            date_str = match.group(1)
            reports.append({
                'file': pdf_file,
                'filename': pdf_file.name,
                'key': date_str,
                'sort_key': date_str
            })
    reports.sort(key=lambda x: x['sort_key'], reverse=True)
    return reports

def generate_summary(pdf_text, report_key, report_name, broker, report_type):
    """Generate summary using Llama 3.1"""
    if not pdf_text.strip():
        return None
    
    if report_type == "STOCK":
        prompt = f"""Summarize this {report_name} report ({report_key}):
1. Revenue Outlook & Growth
2. Key Metrics (margins, ratios)
3. Investment Thesis
4. Risks
5. Business Trend
Keep it concise (150-200 words).
REPORT:
{pdf_text}
Summary:"""
    else:
        prompt = f"""Summarize this {report_name} report ({report_key}):
1. Key Insights & Findings
2. Main Trends Identified
3. Outlook & Expectations
4. Key Risks/Concerns
5. Investment Implications
Keep it concise (150-200 words).
REPORT:
{pdf_text}
Summary:"""
    
    print("  Analyzing...", end="", flush=True)
    summary = call_llama(prompt)
    print(" ✓")
    return summary

# ========================================================================
# START PROCESSING
# ========================================================================

print(f"\n" + "="*90)
print(f"PROCESSING STARTED at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Type: {CONFIG['type']} | Name: {CONFIG['name']} | Broker: {CONFIG['broker']}")
print(f"="*90)

# Find reports based on type
if CONFIG['type'] == "STOCK":
    print(f"\n🔍 Finding {CONFIG['name']} {CONFIG['broker']} quarterly reports...")
    reports = find_stock_reports(CONFIG['name'], CONFIG['broker'])
elif CONFIG['type'] == "SECTOR":
    print(f"\n🔍 Finding {CONFIG['name']} sector reports from {CONFIG['broker']}...")
    reports = find_sector_reports(CONFIG['name'], CONFIG['broker'])
else:  # ECONOMY
    print(f"\n🔍 Finding {CONFIG['name']} reports from {CONFIG['broker']}...")
    reports = find_economy_reports(CONFIG['name'], CONFIG['broker'])

if not reports:
    print(f"❌ No reports found")
else:
    print(f"✓ Found {len(reports)} reports")
    reports_to_use = reports[:CONFIG['qty']]
    print(f"\n📊 Processing {len(reports_to_use)} reports...\n")
    
    summaries = {}
    start_total = time.time()
    
    for idx, report in enumerate(reports_to_use, 1):
        print(f"[{idx}/{len(reports_to_use)}] 📄 {report['key']}: ", end="", flush=True)
        start_time = time.time()
        
        pdf_text = extract_pdf_text(report['file'])
        if pdf_text.strip():
            summary = generate_summary(pdf_text, report['key'], CONFIG['name'], CONFIG['broker'], CONFIG['type'])
            elapsed = int(time.time() - start_time)
            summaries[report['key']] = {
                'filename': report['filename'],
                'summary': summary,
                'time': elapsed
            }
            print(f"({elapsed}s)")
        else:
            print("✗")
    
    print(f"\n✓ Extracted: {len(summaries)} reports")
    
    # ========================================================================
    # GENERATE OUTPUTS
    # ========================================================================
    
    if summaries:
        print("\n" + "="*90)
        print(f"REPORT ANALYSIS: {CONFIG['name']} ({CONFIG['broker']})")
        print(f"Type: {CONFIG['type']} | Latest {len(summaries)} reports")
        print("="*90)
        
        # Show individual summaries
        for idx, (key, data) in enumerate(summaries.items(), 1):
            print(f"\n{'─'*90}")
            print(f"📅 Report {idx}: {key}")
            print(f"{'─'*90}")
            print(data['summary'])
        
        # Master Summary
        print(f"\n\n{'='*90}")
        print(f"MASTER SUMMARY: Overall Trend & Insights")
        print(f"{'='*90}")
        print(f"\n🔄 Generating comprehensive analysis (this may take 10+ minutes)...\n")
        
        all_summaries_text = "\n\n".join([f"{k}:\n{v['summary']}" for k, v in summaries.items()])
        
        if CONFIG['type'] == "STOCK":
            master_prompt = f"""Analyze the trend across these quarterly reports for {CONFIG['name']}:
{all_summaries_text}
Provide:
1. Overall Trend: Accelerating/Stable/Decelerating?
2. Key Changes: Quarter-over-quarter shifts
3. Outlook: What's coming next?
4. Investment Story: Why it matters
Be concise (200-300 words).
Analysis:"""
        else:
            master_prompt = f"""Analyze the trend across these {CONFIG['type'].lower()} reports:
{all_summaries_text}
Provide:
1. Overall Trend: What's the direction?
2. Key Changes: What's shifting?
3. Outlook: What to expect next?
4. Key Implications: Why it matters
Be concise (200-300 words).
Analysis:"""
        
        master_summary = call_llama(master_prompt)
        print(master_summary)
        print(f"\n{'='*90}")
        
        # Performance
        total_time = int(time.time() - start_total)
        print(f"\n{'='*90}")
        print(f"✓ PROCESSING COMPLETE at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*90}")
        print(f"  Total time: {total_time}s (~{total_time//60}m {total_time%60}s)")
        print(f"  Avg per report: {total_time//len(summaries)}s")
        print(f"\n⏭️  Run Cell 4 for final summary!")


PROCESSING STARTED at 2026-04-20 17:17:50
Type: STOCK | Name: ANUP | Broker: AMBIT

🔍 Finding ANUP AMBIT quarterly reports...
✓ Found 8 reports

📊 Processing 3 reports...

[1/3] 📄 FY26Q3:   Analyzing... ✓
(484s)
[2/3] 📄 FY26Q2:   Analyzing... ✓
(431s)
[3/3] 📄 FY26Q1:   Analyzing... ✓
(370s)

✓ Extracted: 3 reports

REPORT ANALYSIS: ANUP (AMBIT)
Type: STOCK | Latest 3 reports

──────────────────────────────────────────────────────────────────────────────────────────
📅 Report 1: FY26Q3
──────────────────────────────────────────────────────────────────────────────────────────
Here's a summary of the ANUP report for FY26Q3 in 150-200 words:

Anup Engineering is expected to meet its revenue growth guidance of 15-20% in FY26 with ~22% EBITDAM. The company is shifting from being product-focused to service-oriented, which should provide stability to the cyclical business. Orders are expected to improve driven by decreasing geopolitical uncertainties, a strong inquiry pipeline, and acceleratio

In [ ]:
print("\n" + "="*90)
print("✓ ANALYSIS COMPLETE")
print("="*90)
if summaries:
    print(f"\n📊 Summary:")
    print(f"  Type: {CONFIG['type']}")
    print(f"  Name: {CONFIG['name']}")
    print(f"  Broker: {CONFIG['broker']}")
    print(f"  Reports: {len(summaries)}")
    print(f"  Status: ✓ Ready for review")
    print(f"\n💡 To analyze different reports/types: Run this notebook again!")